# RUN__pdf_ocr_summary — trạng thái parse báo cáo tài chính, theo cổ phiếu

Đọc **toàn bộ** `raw_data/cafef/financials/statements/**/*.csv` (mọi template: `bank`, `corp`, …)
và tóm tắt thành một bảng, mỗi dòng một cổ phiếu, **5 cột**:

| cột | nghĩa |
|---|---|
| `exchange` | sàn niêm yết, đọc từ chính cột `exchange` của CSV |
| `first_report` | **quý sớm nhất có báo cáo** — quý sớm nhất mà cổ phiếu này có ít nhất một statement `source='pdf'` |
| `balance_sheet` | **quý muộn nhất bị missing** trong bảng cân đối kế toán |
| `income_statement` | quý muộn nhất bị missing trong báo cáo kết quả kinh doanh |
| `cash_flow` | quý muộn nhất bị missing trong báo cáo lưu chuyển tiền tệ |

Quý hiển thị dạng **`2008-Q4`** — dạng sắp xếp được, và đúng dạng `--quarters` /
`QUARTERS` của `pdf_ocr_job` nhận, nên một ô ở bảng này dán thẳng vào lệnh parse được.
(Trên đĩa CSV vẫn ghi `Q4-2008`; cột `period` gốc được giữ nguyên trong `records`.)

Đọc bảng: sau `first_report` thì dữ liệu bắt đầu; **sau quý ghi trong ba cột sau thì statement đó
liền mạch** — không còn quý nào thiếu. Ô `—` nghĩa là statement đó **không thiếu quý nào**.

⚠️ **`missing` là câu trả lời đúng, không phải lỗi.** Theo `CLAUDE.md` §5 rule 24 một số liệu chỉ
được lấy từ PDF gốc; quý nào không có filing hoặc không đọc được thì ghi `missing`. Rất nhiều ô ở
đây là *doanh nghiệp không nộp báo cáo quý đó* (BID/BSR/TCB chỉ nộp báo cáo năm trong các năm đầu),
chứ không phải OCR hỏng.

⚠️ Notebook này **chỉ đọc**, không parse, không OCR, không ghi gì vào `raw_data/`.

## 1 · Định vị thư mục

⚠️ Đường dẫn được dò ngược từ thư mục hiện hành lên tới repo root, nên notebook chạy được cả khi
kernel mở ở `src/kaggle_gpu/` lẫn ở repo root — và **in ra thư mục thật sự đã đọc** (`CWD-1`:
một `STATEMENTS_DIR` tương đối đọc nhầm thư mục rỗng trông y hệt một ticker chưa parse).

In [1]:
from pathlib import Path

import pandas as pd

REPORTS = ["balance_sheet", "income_statement", "cash_flow"]
REL_STATEMENTS = Path("raw_data/cafef/financials/statements")


def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / REL_STATEMENTS).is_dir():
            return candidate
    raise FileNotFoundError(f"khong tim thay {REL_STATEMENTS} tu {here} tro len")


REPO_ROOT = _repo_root()
STATEMENTS_DIR = REPO_ROOT / REL_STATEMENTS

print("cwd            :", Path.cwd())
print("repo root      :", REPO_ROOT)
print("statements dir :", STATEMENTS_DIR)

cwd            : D:\GIT\master-thesis\src\kaggle_gpu
repo root      : D:\GIT\master-thesis
statements dir : D:\GIT\master-thesis\raw_data\cafef\financials\statements


## 2 · Đọc mọi CSV thành một bảng dài

Chỉ lấy các cột meta cần thiết. `encoding='utf-8-sig'` là bắt buộc — cột đầu tiên của các file này
mang BOM, nếu không sẽ thành `﻿symbol`.

In [2]:
META = ["symbol", "exchange", "template", "period", "year", "quarter", "source"]

frames = []
for path in sorted(STATEMENTS_DIR.glob("*/*/*.csv")):
    report = path.parent.name
    if report not in REPORTS:
        print(f"WARNING: bo qua {path} — thu muc bao cao la {report!r}")
        continue
    part = pd.read_csv(path, usecols=META, encoding="utf-8-sig")
    part["report"] = report
    part["file"] = path.relative_to(REPO_ROOT).as_posix()
    frames.append(part)

if not frames:
    raise FileNotFoundError(f"khong co file .csv nao trong {STATEMENTS_DIR}")

records = pd.concat(frames, ignore_index=True)
records["year"] = records["year"].astype(int)
records["quarter"] = records["quarter"].astype(int)
records["source"] = records["source"].fillna("missing")

# mot symbol niem yet tren hai san se pha khoa ticker — doi khoa thay vi im lang gop nham
pairs = records[["exchange", "symbol"]].drop_duplicates()
if pairs["symbol"].duplicated().any():
    records["ticker"] = records["exchange"] + "_" + records["symbol"]
else:
    records["ticker"] = records["symbol"]

# quy -> mot so nguyen tang dan, de lay min/max ma khong sap xep chuoi 'Q4-2008'
records["rank"] = records["year"] * 4 + records["quarter"]

# dang hien thi YYYY-QQ: sap xep duoc, va la dang --quarters cua pdf_ocr_job
records["quarter_id"] = records["year"].astype(str) + "-Q" + records["quarter"].astype(str)

unexpected = sorted(set(records["source"]) - {"pdf", "missing"})
if unexpected:
    print(f"WARNING: gia tri source ngoai du kien: {unexpected} — bang duoi coi chung KHONG phai 'co bao cao'")

print(f"{len(frames)} file / {records['ticker'].nunique()} co phieu / {len(records)} dong")
print(records["source"].value_counts().to_dict())
records

21 file / 7 co phieu / 1182 dong
{'pdf': 971, 'missing': 211}


,symbol,exchange,template,period,year,quarter,source,report,file,ticker,rank,quarter_id
0,ACB,HOSE,bank,Q1-2008,2008,1,missing,balance_sheet,raw_data/cafef/financials/statements/bank/bala...,ACB,8033,2008-Q1
1,ACB,HOSE,bank,Q2-2008,2008,2,missing,balance_sheet,raw_data/cafef/financials/statements/bank/bala...,ACB,8034,2008-Q2
2,ACB,HOSE,bank,Q3-2008,2008,3,missing,balance_sheet,raw_data/cafef/financials/statements/bank/bala...,ACB,8035,2008-Q3
3,ACB,HOSE,bank,Q4-2008,2008,4,missing,balance_sheet,raw_data/cafef/financials/statements/bank/bala...,ACB,8036,2008-Q4
4,ACB,HOSE,bank,Q1-2009,2009,1,missing,balance_sheet,raw_data/cafef/financials/statements/bank/bala...,ACB,8037,2009-Q1
...,...,...,...,...,...,...,...,...,...,...,...,...
1177,VIC,HOSE,corp,Q4-2013,2013,4,missing,income_statement,raw_data/cafef/financials/statements/corp/inco...,VIC,8056,2013-Q4
1178,VIC,HOSE,corp,Q1-2014,2014,1,pdf,income_statement,raw_data/cafef/financials/statements/corp/inco...,VIC,8057,2014-Q1
1179,VIC,HOSE,corp,Q2-2014,2014,2,pdf,income_statement,raw_data/cafef/financials/statements/corp/inco...,VIC,8058,2014-Q2
1180,VIC,HOSE,corp,Q3-2014,2014,3,pdf,income_statement,raw_data/cafef/financials/statements/corp/inco...,VIC,8059,2014-Q3


## 3 · Bảng tóm tắt — 5 cột

`exchange` là sàn của cổ phiếu; `first_report` lấy trên **cả ba** statement (quý sớm nhất có bất kỳ báo cáo nào đọc được);
ba cột sau tính riêng từng statement.

In [3]:
# khoa ticker o cell 2 da bao dam moi ticker chi thuoc mot san
exchange = records.groupby("ticker")["exchange"].first()

parsed = records[records["source"] == "pdf"]
first_report = (
    parsed.loc[parsed.groupby("ticker")["rank"].idxmin()]
    .set_index("ticker")["quarter_id"]
    .rename("first_report")
)

missing = records[records["source"] == "missing"]
last_missing = (
    missing.loc[missing.groupby(["ticker", "report"])["rank"].idxmax()]
    .pivot(index="ticker", columns="report", values="quarter_id")
    .reindex(columns=REPORTS)
)

summary = (
    pd.DataFrame(index=pd.Index(sorted(records["ticker"].unique()), name="ticker"))
    .join(exchange)
    .join(first_report)
    .join(last_missing)
    .fillna("—")
)
summary

,exchange,first_report,balance_sheet,income_statement,cash_flow
ticker,,,,,
ACB,HOSE,2009-Q2,2009-Q3,2009-Q4,2009-Q3
BID,HOSE,2008-Q4,2011-Q2,2011-Q4,2011-Q2
BSR,HOSE,2016-Q4,2019-Q4,2020-Q4,2018-Q2
CTG,HOSE,2008-Q4,2025-Q1,2025-Q4,2024-Q1
TCB,HOSE,2009-Q4,2013-Q1,2012-Q3,2021-Q1
VCB,HOSE,2008-Q4,2009-Q2,2008-Q4,2009-Q2
VIC,HOSE,2008-Q2,2010-Q4,2013-Q4,2014-Q1


## 4 · Phụ — đếm quý đã parse / còn thiếu

Không nằm trong 4 cột được hỏi, nhưng là mẫu số để đọc bảng trên: một `first_report` sớm mà
`coverage` thấp nghĩa là chuỗi dài nhưng rỗng, chứ không phải lịch sử dài.

⚠️ `quarters` là số dòng CSV, tức khoảng thời gian file phủ — **không** phải số quý doanh
nghiệp thực sự nộp báo cáo. Quý không có filing nào cũng nằm trong mẫu số này.

In [4]:
tally = (
    records.groupby(["ticker", "report"])["source"]
    .value_counts()
    .unstack("source")
    .reindex(columns=["pdf", "missing"])
    .fillna(0)
    .astype(int)
)
tally["quarters"] = tally["pdf"] + tally["missing"]
tally["coverage"] = (tally["pdf"] / tally["quarters"]).round(3)
tally

source                   pdf  missing  quarters  coverage
ticker report                                            
ACB    balance_sheet      67        6        73     0.918
       cash_flow          66        7        73     0.904
       income_statement   66        7        73     0.904
BID    balance_sheet      62        8        70     0.886
       cash_flow          61        9        70     0.871
       income_statement   57       13        70     0.814
BSR    balance_sheet      10        7        17     0.588
       cash_flow          13        4        17     0.765
       income_statement    6       11        17     0.353
CTG    balance_sheet      31       39        70     0.443
       cash_flow          61        9        70     0.871
       income_statement   35       35        70     0.500
TCB    balance_sheet      57       10        67     0.851
       cash_flow          52       15        67     0.776
       income_statement   59        8        67     0.881
VCB    balance_sheet      68        2        70     0.971
       cash_flow          69        1        70     0.986
       income_statement   69        1        70     0.986
VIC    balance_sheet      20        7        27     0.741
       cash_flow          20        7        27     0.741
       income_statement   22        5        27     0.815